In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path] + list(current_path.parents)
        if (path / "src").is_dir() and (path / "models").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the Fraud-detection-ML-V2 project root."
    )

SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [PROJECT_ROOT, SRC_DIR]:
    path_string = str(path)
    if path_string not in sys.path:
        sys.path.insert(0, path_string)

required_files = [
    SRC_DIR / "detection_service.py",
    SRC_DIR / "feature_contract.py",
    SRC_DIR / "ml_inference.py",
    SRC_DIR / "ml_service.py",
    SRC_DIR / "rule_engine.py",
    SRC_DIR / "decision_engine.py",
    SRC_DIR / "shap_explainer.py",
    MODELS_DIR / "xgboost_fraud_detector.json",
    MODELS_DIR / "feature_columns.json",
    MODELS_DIR / "model_info.json",
    MODELS_DIR / "deployment_config.json",
    MODELS_DIR / "rule_engine_config.json",
    MODELS_DIR / "decision_engine_config.json",
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required project files are missing:\n"
        + "\n".join(missing_files)
    )

print("PHASE 13 PROJECT CHECK")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"Models directory: {MODELS_DIR}")
print("=" * 60)
print("Project structure check: PASS")

PHASE 13 PROJECT CHECK
Project root: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2
Source directory: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\src
Models directory: C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\models
Project structure check: PASS


In [2]:
import inspect
import importlib

detection_service = importlib.import_module("detection_service")

public_functions = {
    name: obj
    for name, obj in inspect.getmembers(
        detection_service,
        inspect.isfunction
    )
    if not name.startswith("_")
}

print("DETECTION SERVICE FUNCTIONS")
print("=" * 60)

for name, function_object in public_functions.items():
    try:
        signature = inspect.signature(function_object)
    except Exception:
        signature = "signature unavailable"

    print(f"{name}{signature}")

print("=" * 60)
print("Detection service import: PASS")

DETECTION SERVICE FUNCTIONS
evaluate_rules(transaction)
explain_transaction(transaction, top_n=3)
feature_reason_text(feature, value, shap_value)
generate_reason(rule_flags, ml_score, shap_result, decision)
get_ml_score(transaction)
make_decision(ml_fraud_score, rule_result)
score_transaction(transaction)
Detection service import: PASS


In [3]:
preferred_names = [
    "detect_transaction",
    "detect",
    "process_transaction",
    "run_detection",
    "process_detection"
]

detection_function = None

for name in preferred_names:
    if name in public_functions:
        detection_function = public_functions[name]
        break

if detection_function is None:
    candidates = [
        function_object
        for name, function_object in public_functions.items()
        if any(
            keyword in name.lower()
            for keyword in ["detect", "process", "score"]
        )
    ]

    if candidates:
        detection_function = candidates[0]

if detection_function is None:
    raise RuntimeError(
        "Could not identify the production detection function "
        "inside src/detection_service.py."
    )

print("PRODUCTION ENTRY POINT")
print("=" * 60)
print(f"Function: {detection_function.__name__}")

try:
    print(f"Signature: {inspect.signature(detection_function)}")
except Exception:
    print("Signature: unavailable")

print("=" * 60)
print("Detection entry point: READY")

PRODUCTION ENTRY POINT
Function: get_ml_score
Signature: (transaction)
Detection entry point: READY


In [4]:
baseline_input = {
    "transaction_id": "ROBUSTNESS_BASELINE_000001",
    "user_id": "USER_ROBUSTNESS_000001",
    "amount": 100.0,
    "amount_vs_avg_ratio": 1.0,
    "txn_count_last_5min": 0,
    "time_since_last_txn_sec": 600.0,
    "distance_from_last_location_km": 0.0,
    "merchant_category_is_new_for_user": 0
}

print("BASELINE FEATURE VECTOR")
print("=" * 60)

for key, value in baseline_input.items():
    print(f"{key}: {value}")

print("=" * 60)
print("Baseline input created: READY")

BASELINE FEATURE VECTOR
transaction_id: ROBUSTNESS_BASELINE_000001
user_id: USER_ROBUSTNESS_000001
amount: 100.0
amount_vs_avg_ratio: 1.0
txn_count_last_5min: 0
time_since_last_txn_sec: 600.0
distance_from_last_location_km: 0.0
merchant_category_is_new_for_user: 0
Baseline input created: READY


In [5]:
import inspect

def call_detection(function_object, input_data):
    signature = inspect.signature(function_object)
    parameters = list(signature.parameters.values())

    usable_parameters = [
        parameter
        for parameter in parameters
        if parameter.kind not in (
            inspect.Parameter.VAR_POSITIONAL,
            inspect.Parameter.VAR_KEYWORD
        )
    ]

    if len(usable_parameters) == 1:
        parameter = usable_parameters[0]

        if parameter.name in input_data:
            return function_object(
                input_data[parameter.name]
            )

        return function_object(
            input_data
        )

    kwargs = {}

    for parameter in usable_parameters:
        if parameter.name in input_data:
            kwargs[parameter.name] = input_data[parameter.name]
        elif parameter.default is inspect.Parameter.empty:
            raise ValueError(
                f"Required parameter '{parameter.name}' "
                f"cannot be supplied from the Feature Vector."
            )

    return function_object(**kwargs)

print("Universal detection caller: READY")

Universal detection caller: READY


In [6]:
baseline_output = call_detection(
    detection_function,
    baseline_input
)

print("BASELINE DETECTION TEST")
print("=" * 60)
print("Status: PASS")
print(f"Output type: {type(baseline_output).__name__}")

if isinstance(baseline_output, dict):
    print("Output fields:")
    for key in baseline_output:
        print(f"  - {key}")
else:
    print("Output:")
    print(baseline_output)

print("=" * 60)

BASELINE DETECTION TEST
Status: PASS
Output type: dict
Output fields:
  - ml_score
  - ml_prediction
  - ml_label
  - model
  - model_version


In [7]:
invalid_input_tests = [
    ("None input", None),
    ("Empty dictionary", {}),
    ("Empty list", []),
    ("Empty string", ""),
    ("Integer input", 12345),
    ("Boolean input", True)
]

invalid_results = []

for test_name, invalid_input in invalid_input_tests:
    try:
        output = call_detection(
            detection_function,
            invalid_input
        )

        invalid_results.append({
            "test": test_name,
            "status": "UNEXPECTED_ACCEPTANCE",
            "output_type": type(output).__name__,
            "error": ""
        })

    except Exception as exc:
        invalid_results.append({
            "test": test_name,
            "status": "REJECTED",
            "output_type": "",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in invalid_results:
    print(
        f"{result['test']}: "
        f"{result['status']}"
    )

print("=" * 60)
print("Invalid-input execution completed.")

None input: REJECTED
Empty dictionary: REJECTED
Empty list: REJECTED
Empty string: REJECTED
Integer input: REJECTED
Boolean input: REJECTED
Invalid-input execution completed.


In [8]:
required_feature_fields = [
    "transaction_id",
    "user_id",
    "amount",
    "amount_vs_avg_ratio",
    "txn_count_last_5min",
    "time_since_last_txn_sec",
    "distance_from_last_location_km",
    "merchant_category_is_new_for_user"
]

missing_field_results = []

for field in required_feature_fields:
    test_input = baseline_input.copy()
    test_input.pop(field)

    try:
        output = call_detection(
            detection_function,
            test_input
        )

        missing_field_results.append({
            "field": field,
            "status": "UNEXPECTED_ACCEPTANCE"
        })

    except Exception as exc:
        missing_field_results.append({
            "field": field,
            "status": "REJECTED",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in missing_field_results:
    print(
        f"Missing {result['field']}: "
        f"{result['status']}"
    )

print("=" * 60)
print("Missing-field testing completed.")

Missing transaction_id: UNEXPECTED_ACCEPTANCE
Missing user_id: UNEXPECTED_ACCEPTANCE
Missing amount: REJECTED
Missing amount_vs_avg_ratio: REJECTED
Missing txn_count_last_5min: REJECTED
Missing time_since_last_txn_sec: REJECTED
Missing distance_from_last_location_km: REJECTED
Missing merchant_category_is_new_for_user: REJECTED
Missing-field testing completed.


In [9]:
type_tests = [
    ("amount", "invalid"),
    ("amount_vs_avg_ratio", "invalid"),
    ("txn_count_last_5min", "invalid"),
    ("time_since_last_txn_sec", "invalid"),
    ("distance_from_last_location_km", "invalid"),
    ("merchant_category_is_new_for_user", "invalid")
]

type_test_results = []

for field, invalid_value in type_tests:
    test_input = baseline_input.copy()
    test_input[field] = invalid_value

    try:
        output = call_detection(
            detection_function,
            test_input
        )

        type_test_results.append({
            "field": field,
            "status": "UNEXPECTED_ACCEPTANCE"
        })

    except Exception as exc:
        type_test_results.append({
            "field": field,
            "status": "REJECTED",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in type_test_results:
    print(
        f"{result['field']}: "
        f"{result['status']}"
    )

print("=" * 60)
print("Invalid-type testing completed.")

amount: REJECTED
amount_vs_avg_ratio: REJECTED
txn_count_last_5min: REJECTED
time_since_last_txn_sec: REJECTED
distance_from_last_location_km: REJECTED
merchant_category_is_new_for_user: REJECTED
Invalid-type testing completed.


In [10]:
import math

special_value_tests = [
    ("amount_nan", "amount", float("nan")),
    ("amount_positive_infinity", "amount", float("inf")),
    ("amount_negative_infinity", "amount", float("-inf")),
    ("ratio_nan", "amount_vs_avg_ratio", float("nan")),
    ("ratio_positive_infinity", "amount_vs_avg_ratio", float("inf")),
    ("distance_nan", "distance_from_last_location_km", float("nan")),
    ("distance_positive_infinity", "distance_from_last_location_km", float("inf"))
]

special_value_results = []

for test_name, field, value in special_value_tests:
    test_input = baseline_input.copy()
    test_input[field] = value

    try:
        output = call_detection(
            detection_function,
            test_input
        )

        special_value_results.append({
            "test": test_name,
            "status": "PROCESSED"
        })

    except Exception as exc:
        special_value_results.append({
            "test": test_name,
            "status": "REJECTED",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in special_value_results:
    print(
        f"{result['test']}: "
        f"{result['status']}"
    )

print("=" * 60)
print("Special numeric-value testing completed.")

amount_nan: REJECTED
amount_positive_infinity: REJECTED
amount_negative_infinity: REJECTED
ratio_nan: REJECTED
ratio_positive_infinity: REJECTED
distance_nan: REJECTED
distance_positive_infinity: REJECTED
Special numeric-value testing completed.


In [11]:
boundary_cases = {
    "minimum_reasonable_values": {
        **baseline_input,
        "transaction_id": "BOUNDARY_MIN",
        "amount": 0.01,
        "amount_vs_avg_ratio": 0.01,
        "txn_count_last_5min": 0,
        "time_since_last_txn_sec": 0.01,
        "distance_from_last_location_km": 0.0,
        "merchant_category_is_new_for_user": 0
    },
    "normal_values": {
        **baseline_input,
        "transaction_id": "BOUNDARY_NORMAL",
        "amount": 100.0,
        "amount_vs_avg_ratio": 1.0,
        "txn_count_last_5min": 1,
        "time_since_last_txn_sec": 600.0,
        "distance_from_last_location_km": 10.0,
        "merchant_category_is_new_for_user": 0
    },
    "high_values": {
        **baseline_input,
        "transaction_id": "BOUNDARY_HIGH",
        "amount": 100000.0,
        "amount_vs_avg_ratio": 100.0,
        "txn_count_last_5min": 100,
        "time_since_last_txn_sec": 1.0,
        "distance_from_last_location_km": 10000.0,
        "merchant_category_is_new_for_user": 1
    }
}

for name, test_input in boundary_cases.items():
    print(name)
    print("-" * 40)

    for key, value in test_input.items():
        print(f"{key}: {value}")

    print()

minimum_reasonable_values
----------------------------------------
transaction_id: BOUNDARY_MIN
user_id: USER_ROBUSTNESS_000001
amount: 0.01
amount_vs_avg_ratio: 0.01
txn_count_last_5min: 0
time_since_last_txn_sec: 0.01
distance_from_last_location_km: 0.0
merchant_category_is_new_for_user: 0

normal_values
----------------------------------------
transaction_id: BOUNDARY_NORMAL
user_id: USER_ROBUSTNESS_000001
amount: 100.0
amount_vs_avg_ratio: 1.0
txn_count_last_5min: 1
time_since_last_txn_sec: 600.0
distance_from_last_location_km: 10.0
merchant_category_is_new_for_user: 0

high_values
----------------------------------------
transaction_id: BOUNDARY_HIGH
user_id: USER_ROBUSTNESS_000001
amount: 100000.0
amount_vs_avg_ratio: 100.0
txn_count_last_5min: 100
time_since_last_txn_sec: 1.0
distance_from_last_location_km: 10000.0
merchant_category_is_new_for_user: 1



In [12]:
boundary_results = []

for case_name, test_input in boundary_cases.items():
    try:
        output = call_detection(
            detection_function,
            test_input
        )

        boundary_results.append({
            "case": case_name,
            "status": "PASS",
            "output_type": type(output).__name__
        })

    except Exception as exc:
        boundary_results.append({
            "case": case_name,
            "status": "FAIL",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in boundary_results:
    print(
        f"{result['case']}: "
        f"{result['status']}"
    )

print("=" * 60)

minimum_reasonable_values: PASS
normal_values: PASS
high_values: PASS


In [13]:
rule_scenarios = {
    "normal_transaction": {
        **baseline_input,
        "transaction_id": "RULE_NORMAL",
        "amount_vs_avg_ratio": 1.0,
        "txn_count_last_5min": 0,
        "time_since_last_txn_sec": 600.0,
        "distance_from_last_location_km": 10.0,
        "merchant_category_is_new_for_user": 0
    },
    "high_amount": {
        **baseline_input,
        "transaction_id": "RULE_HIGH_AMOUNT",
        "amount": 1000.0,
        "amount_vs_avg_ratio": 4.0
    },
    "high_velocity": {
        **baseline_input,
        "transaction_id": "RULE_HIGH_VELOCITY",
        "txn_count_last_5min": 10
    },
    "impossible_travel": {
        **baseline_input,
        "transaction_id": "RULE_IMPOSSIBLE_TRAVEL",
        "time_since_last_txn_sec": 60.0,
        "distance_from_last_location_km": 1000.0
    },
    "new_merchant_category": {
        **baseline_input,
        "transaction_id": "RULE_NEW_CATEGORY",
        "merchant_category_is_new_for_user": 1
    },
    "multiple_signals": {
        **baseline_input,
        "transaction_id": "RULE_MULTIPLE",
        "amount": 5000.0,
        "amount_vs_avg_ratio": 10.0,
        "txn_count_last_5min": 20,
        "time_since_last_txn_sec": 30.0,
        "distance_from_last_location_km": 1500.0,
        "merchant_category_is_new_for_user": 1
    }
}

rule_scenario_results = []

for scenario_name, test_input in rule_scenarios.items():
    try:
        output = call_detection(
            detection_function,
            test_input
        )

        rule_scenario_results.append({
            "scenario": scenario_name,
            "status": "PASS",
            "output_type": type(output).__name__
        })

    except Exception as exc:
        rule_scenario_results.append({
            "scenario": scenario_name,
            "status": "FAIL",
            "error": f"{type(exc).__name__}: {exc}"
        })

for result in rule_scenario_results:
    print(
        f"{result['scenario']}: "
        f"{result['status']}"
    )

print("=" * 60)
print("Rule-oriented scenario testing completed.")

normal_transaction: PASS
high_amount: PASS
high_velocity: PASS
impossible_travel: PASS
new_merchant_category: PASS
multiple_signals: PASS
Rule-oriented scenario testing completed.


In [14]:
stress_size = 5000

stress_inputs = []

for i in range(stress_size):
    transaction = baseline_input.copy()

    transaction["transaction_id"] = f"STRESS_{i:06d}"
    transaction["user_id"] = f"USER_{i % 100:04d}"
    transaction["amount"] = float(
        50.0 + ((i * 37) % 5000)
    )
    transaction["amount_vs_avg_ratio"] = float(
        0.5 + ((i * 13) % 100) / 10
    )
    transaction["txn_count_last_5min"] = int(
        (i * 7) % 15
    )
    transaction["time_since_last_txn_sec"] = float(
        1 + ((i * 17) % 3600)
    )
    transaction["distance_from_last_location_km"] = float(
        (i * 19) % 2000
    )
    transaction["merchant_category_is_new_for_user"] = int(
        i % 2
    )

    stress_inputs.append(transaction)

print("STRESS TEST DATASET")
print("=" * 60)
print(f"Transactions generated: {len(stress_inputs)}")
print("=" * 60)

STRESS TEST DATASET
Transactions generated: 5000


In [15]:
import time

stress_success = 0
stress_failures = 0
stress_errors = []

stress_start = time.perf_counter()

for transaction in stress_inputs:
    try:
        call_detection(
            detection_function,
            transaction
        )
        stress_success += 1

    except Exception as exc:
        stress_failures += 1

        if len(stress_errors) < 20:
            stress_errors.append({
                "transaction_id": transaction["transaction_id"],
                "error": f"{type(exc).__name__}: {exc}"
            })

stress_end = time.perf_counter()

stress_total_time_sec = stress_end - stress_start

stress_throughput = (
    stress_success / stress_total_time_sec
    if stress_total_time_sec > 0
    else 0.0
)

print("STRESS TEST RESULTS")
print("=" * 60)
print(f"Total transactions: {stress_size}")
print(f"Successful: {stress_success}")
print(f"Failures: {stress_failures}")
print(f"Total time: {stress_total_time_sec:.4f} seconds")
print(f"Throughput: {stress_throughput:.2f} transactions/second")
print("=" * 60)

if stress_errors:
    print("First failures:")
    for error in stress_errors:
        print(error)

STRESS TEST RESULTS
Total transactions: 5000
Successful: 5000
Failures: 0
Total time: 20.6829 seconds
Throughput: 241.75 transactions/second


In [16]:
expected_stress_count = len(stress_inputs)
actual_stress_count = stress_success + stress_failures

print("STRESS TEST VALIDATION")
print("=" * 60)
print(f"Expected transactions: {expected_stress_count}")
print(f"Processed transactions: {actual_stress_count}")

if actual_stress_count != expected_stress_count:
    raise RuntimeError(
        "Stress test accounting mismatch."
    )

print("Accounting check: PASS")

STRESS TEST VALIDATION
Expected transactions: 5000
Processed transactions: 5000
Accounting check: PASS


In [17]:
regression_cases = [
    {
        "case_id": "REG_001_NORMAL",
        **baseline_input
    },
    {
        "case_id": "REG_002_HIGH_AMOUNT",
        **baseline_input,
        "transaction_id": "REG_002_HIGH_AMOUNT",
        "amount": 1000.0,
        "amount_vs_avg_ratio": 4.0
    },
    {
        "case_id": "REG_003_HIGH_VELOCITY",
        **baseline_input,
        "transaction_id": "REG_003_HIGH_VELOCITY",
        "txn_count_last_5min": 10
    },
    {
        "case_id": "REG_004_IMPOSSIBLE_TRAVEL",
        **baseline_input,
        "transaction_id": "REG_004_IMPOSSIBLE_TRAVEL",
        "distance_from_last_location_km": 1000.0,
        "time_since_last_txn_sec": 60.0
    },
    {
        "case_id": "REG_005_NEW_CATEGORY",
        **baseline_input,
        "transaction_id": "REG_005_NEW_CATEGORY",
        "merchant_category_is_new_for_user": 1
    },
    {
        "case_id": "REG_006_MULTIPLE_SIGNALS",
        **baseline_input,
        "transaction_id": "REG_006_MULTIPLE_SIGNALS",
        "amount": 5000.0,
        "amount_vs_avg_ratio": 10.0,
        "txn_count_last_5min": 20,
        "distance_from_last_location_km": 1500.0,
        "time_since_last_txn_sec": 30.0,
        "merchant_category_is_new_for_user": 1
    }
]

print("Regression cases created:")
print(len(regression_cases))

Regression cases created:
6


In [18]:
from pathlib import Path
import json
import datetime

REGRESSION_DIR = PROJECT_ROOT / "outputs" / "reports"
REGRESSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REGRESSION_FILE = (
    REGRESSION_DIR /
    "phase13_regression_snapshot.json"
)

current_results = []

for case in regression_cases:
    try:
        output = call_detection(
            detection_function,
            case
        )

        current_results.append({
            "case_id": case["case_id"],
            "status": "SUCCESS",
            "output": output
        })

    except Exception as exc:
        current_results.append({
            "case_id": case["case_id"],
            "status": "ERROR",
            "output": None,
            "error": f"{type(exc).__name__}: {exc}"
        })

if not REGRESSION_FILE.exists():
    snapshot = {
        "created_at": datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat(),
        "cases": current_results
    }

    with open(
        REGRESSION_FILE,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            snapshot,
            file,
            indent=2,
            default=str
        )

    print("Regression snapshot created.")
    print(REGRESSION_FILE)

else:
    with open(
        REGRESSION_FILE,
        "r",
        encoding="utf-8"
    ) as file:
        previous_snapshot = json.load(file)

    previous_cases = {
        case["case_id"]: case
        for case in previous_snapshot.get("cases", [])
    }

    regression_differences = []

    for current_case in current_results:
        case_id = current_case["case_id"]

        previous_case = previous_cases.get(case_id)

        if previous_case is None:
            regression_differences.append({
                "case_id": case_id,
                "difference": "New regression case"
            })
            continue

        previous_status = previous_case.get("status")
        current_status = current_case.get("status")

        if previous_status != current_status:
            regression_differences.append({
                "case_id": case_id,
                "difference": (
                    f"Status changed from "
                    f"{previous_status} to {current_status}"
                )
            })

        previous_output = json.dumps(
            previous_case.get("output"),
            sort_keys=True,
            default=str
        )

        current_output = json.dumps(
            current_case.get("output"),
            sort_keys=True,
            default=str
        )

        if previous_output != current_output:
            regression_differences.append({
                "case_id": case_id,
                "difference": "Output changed"
            })

    print("REGRESSION TEST")
    print("=" * 60)
    print(f"Cases checked: {len(current_results)}")
    print(f"Differences: {len(regression_differences)}")

    if regression_differences:
        for difference in regression_differences:
            print(difference)
    else:
        print("Regression comparison: PASS")

Regression snapshot created.
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\reports\phase13_regression_snapshot.json


In [19]:
invalid_test_count = sum(
    1
    for result in invalid_results
    if result["status"] == "REJECTED"
)

missing_field_test_count = sum(
    1
    for result in missing_field_results
    if result["status"] == "REJECTED"
)

type_test_count = sum(
    1
    for result in type_test_results
    if result["status"] == "REJECTED"
)

boundary_pass_count = sum(
    1
    for result in boundary_results
    if result["status"] == "PASS"
)

rule_scenario_pass_count = sum(
    1
    for result in rule_scenario_results
    if result["status"] == "PASS"
)

regression_differences = globals().get(
    "regression_differences",
    []
)

checks = {
    "baseline_detection_passed": "baseline_output" in globals(),
    "invalid_input_tests_completed": len(invalid_results) == 6,
    "missing_field_tests_completed": len(missing_field_results) == 8,
    "invalid_type_tests_completed": len(type_test_results) == 6,
    "boundary_tests_completed": len(boundary_results) == 3,
    "rule_scenarios_completed": len(rule_scenario_results) == 6,
    "stress_test_completed": (
        stress_success + stress_failures == stress_size
    ),
    "regression_tests_completed": len(current_results) == 6,
}

overall = all(checks.values())

print("PHASE 13 ROBUSTNESS & TESTING")
print("=" * 60)

for name, status in checks.items():
    print(f"{name}: {status}")

print("=" * 60)

print(f"Invalid inputs rejected: {invalid_test_count}")
print(f"Missing-field inputs rejected: {missing_field_test_count}")
print(f"Invalid-type inputs rejected: {type_test_count}")
print(f"Boundary scenarios passed: {boundary_pass_count}/{len(boundary_results)}")
print(f"Rule scenarios passed: {rule_scenario_pass_count}/{len(rule_scenario_results)}")
print(f"Stress successful: {stress_success}/{stress_size}")
print(f"Stress failures: {stress_failures}")

if regression_differences:
    print(f"Regression differences: {len(regression_differences)}")
else:
    print("Regression differences: 0")

print("=" * 60)
print(f"Overall Phase 13 execution: {overall}")

PHASE 13 ROBUSTNESS & TESTING
baseline_detection_passed: True
invalid_input_tests_completed: True
missing_field_tests_completed: True
invalid_type_tests_completed: True
boundary_tests_completed: True
rule_scenarios_completed: True
stress_test_completed: True
regression_tests_completed: True
Invalid inputs rejected: 6
Missing-field inputs rejected: 6
Invalid-type inputs rejected: 6
Boundary scenarios passed: 3/3
Rule scenarios passed: 6/6
Stress successful: 5000/5000
Stress failures: 0
Regression differences: 0
Overall Phase 13 execution: True


In [20]:
from pathlib import Path
import json
import datetime

REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

phase13_report = {
    "phase": 13,
    "title": "Robustness & Testing",
    "project_version": "V2.1",
    "timestamp_utc": datetime.datetime.now(
        datetime.timezone.utc
    ).isoformat(),
    "test_categories": {
        "invalid_inputs": invalid_results,
        "missing_fields": missing_field_results,
        "invalid_types": type_test_results,
        "special_numeric_values": special_value_results,
        "boundary_tests": boundary_results,
        "rule_scenarios": rule_scenario_results,
        "stress_testing": {
            "total": stress_size,
            "successful": stress_success,
            "failures": stress_failures,
            "total_time_sec": stress_total_time_sec,
            "throughput_transactions_per_second": stress_throughput,
            "errors": stress_errors
        },
        "regression": {
            "cases": current_results,
            "differences": regression_differences
        }
    },
    "verification": checks
}

report_path = (
    REPORT_DIR /
    "phase13_robustness_testing_report.json"
)

metrics_path = (
    METRICS_DIR /
    "phase13_robustness_testing_metrics.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        phase13_report,
        file,
        indent=2,
        default=str
    )

with open(
    metrics_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            "phase": 13,
            "invalid_inputs_tested": len(invalid_results),
            "missing_fields_tested": len(missing_field_results),
            "invalid_types_tested": len(type_test_results),
            "special_numeric_values_tested": len(special_value_results),
            "boundary_cases_tested": len(boundary_results),
            "rule_scenarios_tested": len(rule_scenario_results),
            "stress_transactions": stress_size,
            "stress_success": stress_success,
            "stress_failures": stress_failures,
            "stress_throughput_transactions_per_second": stress_throughput,
            "regression_cases": len(current_results),
            "regression_differences": len(regression_differences)
        },
        file,
        indent=2,
        default=str
    )

print("PHASE 13 REPORTS SAVED")
print("=" * 60)
print(report_path)
print(metrics_path)
print("=" * 60)

PHASE 13 REPORTS SAVED
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\reports\phase13_robustness_testing_report.json
C:\Users\Lenovo\Desktop\Fraud-detection-ML-V2\outputs\metrics\phase13_robustness_testing_metrics.json


In [21]:
from pathlib import Path

report_path = (
    PROJECT_ROOT /
    "outputs" /
    "reports" /
    "phase13_robustness_testing_report.json"
)

metrics_path = (
    PROJECT_ROOT /
    "outputs" /
    "metrics" /
    "phase13_robustness_testing_metrics.json"
)

final_checks = {
    "baseline_test": "baseline_output" in globals(),
    "invalid_inputs": len(invalid_results) == 6,
    "missing_fields": len(missing_field_results) == 8,
    "invalid_types": len(type_test_results) == 6,
    "special_values": len(special_value_results) == 7,
    "boundary_testing": len(boundary_results) == 3,
    "rule_testing": len(rule_scenario_results) == 6,
    "stress_testing": (
        stress_success + stress_failures == stress_size
    ),
    "regression_testing": len(current_results) == 6,
    "report_saved": report_path.exists(),
    "metrics_saved": metrics_path.exists()
}

final_status = all(final_checks.values())

print("PHASE 13 FINAL STATUS")
print("=" * 60)

for name, status in final_checks.items():
    print(f"{name}: {status}")

print("=" * 60)
print(f"Overall verification: {final_status}")

if final_status:
    print("PHASE 13 STATUS: READY")
else:
    print("PHASE 13 STATUS: REVIEW REQUIRED")

PHASE 13 FINAL STATUS
baseline_test: True
invalid_inputs: True
missing_fields: True
invalid_types: True
special_values: True
boundary_testing: True
rule_testing: True
stress_testing: True
regression_testing: True
report_saved: True
metrics_saved: True
Overall verification: True
PHASE 13 STATUS: READY
